# E-commerce Sales Analysis with SQL

## 1. Business Context

Olist is a Brazilian marketplace that connects small retailers to
customers across the country. This analysis explores its public dataset —
**~100,000 real orders (2016–2018)** spread across 9 relational tables —
using SQL (DuckDB) to answer four business questions:

1. **How have sales evolved over time, and is there seasonality?**
2. **Which product categories drive revenue?**
3. **Who are the most valuable customers, and where are they located?**
4. **Does delivery performance affect customer satisfaction?**

All queries are written in standard analytical SQL — joins across
multiple tables, CTEs, aggregations and window functions — with results
and business interpretation shown after each query.

## 2. Setup & Data Loading

In [11]:
# Mount Google Drive to access the dataset
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Install and set up DuckDB (analytical SQL engine)
%pip install duckdb --quiet

import duckdb
import pandas as pd

con = duckdb.connect()

In [ ]:
# Register each CSV file as a SQL view
path = "/content/drive/MyDrive/olist_data/"

tables = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

for name, file in tables.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_csv_auto('{path}{file}')")

print("Tables registered:", ", ".join(tables.keys()))

## 3. Database Overview

In [ ]:
# Row counts per table
con.execute("""
    SELECT 'orders' AS table_name, COUNT(*) AS rows FROM orders
    UNION ALL SELECT 'order_items', COUNT(*) FROM order_items
    UNION ALL SELECT 'customers', COUNT(*) FROM customers
    UNION ALL SELECT 'products', COUNT(*) FROM products
    UNION ALL SELECT 'payments', COUNT(*) FROM payments
    UNION ALL SELECT 'reviews', COUNT(*) FROM reviews
    ORDER BY rows DESC
""").df()

In [ ]:
# First look at the orders table
con.execute("SELECT * FROM orders LIMIT 5").df()

## 4. Business Questions & SQL Analysis

### 4.1 How have sales evolved over time?

In [ ]:
# Monthly revenue and order volume (delivered orders only)
con.execute("""
    WITH monthly_sales AS (
        SELECT
            DATE_TRUNC('month', o.order_purchase_timestamp) AS month,
            COUNT(DISTINCT o.order_id) AS total_orders,
            ROUND(SUM(oi.price + oi.freight_value), 2) AS total_revenue
        FROM orders o
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        month,
        total_orders,
        total_revenue,
        ROUND(100.0 * (total_revenue - LAG(total_revenue) OVER (ORDER BY month))
              / LAG(total_revenue) OVER (ORDER BY month), 1) AS revenue_growth_pct
    FROM monthly_sales
    WHERE month BETWEEN '2017-01-01' AND '2018-08-01'
    ORDER BY month
""").df()

**Reading the results:** Revenue grows steadily through 2017, with a
dramatic spike in **November 2017 (+49% MoM)** — Black Friday. After the
holiday season, sales stabilize at a new, higher level in 2018,
suggesting the promotional peak also acquired customers who kept buying.
*Note: 2016 and post-August 2018 data are excluded as incomplete periods.*

### 4.2 Which product categories drive revenue?

In [ ]:
# Top 10 categories by revenue, with share of total
con.execute("""
    WITH category_revenue AS (
        SELECT
            COALESCE(ct.product_category_name_english, p.product_category_name) AS category,
            ROUND(SUM(oi.price), 2) AS revenue,
            COUNT(DISTINCT oi.order_id) AS orders,
            ROUND(AVG(oi.price), 2) AS avg_item_price
        FROM order_items oi
        JOIN products p ON oi.product_id = p.product_id
        LEFT JOIN category_translation ct
               ON p.product_category_name = ct.product_category_name
        JOIN orders o ON oi.order_id = o.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        category,
        revenue,
        orders,
        avg_item_price,
        ROUND(100.0 * revenue / SUM(revenue) OVER (), 1) AS pct_of_total
    FROM category_revenue
    ORDER BY revenue DESC
    LIMIT 10
""").df()

**Reading the results:** Revenue is fragmented — the top category
(health & beauty) holds only ~10% of total revenue, and the top 10
together account for roughly 60%. This is a long-tail marketplace:
growth strategies should combine strengthening top categories with
better discovery for niche ones. Note the contrast between high-volume /
low-price categories (bed & bath) and low-volume / high-ticket ones
(watches & gifts).

### 4.3 Who are the most valuable customers?

In [ ]:
# Top 10 customers by lifetime spend
con.execute("""
    SELECT
        c.customer_unique_id,
        c.customer_city,
        c.customer_state,
        COUNT(DISTINCT o.order_id) AS orders,
        ROUND(SUM(oi.price + oi.freight_value), 2) AS lifetime_spend
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY 1, 2, 3
    ORDER BY lifetime_spend DESC
    LIMIT 10
""").df()

In [ ]:
# Revenue by state, ranked with a window function
con.execute("""
    WITH state_revenue AS (
        SELECT
            c.customer_state,
            ROUND(SUM(oi.price + oi.freight_value), 2) AS revenue,
            COUNT(DISTINCT c.customer_unique_id) AS customers
        FROM customers c
        JOIN orders o ON c.customer_id = o.customer_id
        JOIN order_items oi ON o.order_id = oi.order_id
        WHERE o.order_status = 'delivered'
        GROUP BY 1
    )
    SELECT
        RANK() OVER (ORDER BY revenue DESC) AS rank,
        customer_state,
        revenue,
        customers,
        ROUND(revenue / customers, 2) AS revenue_per_customer
    FROM state_revenue
    ORDER BY revenue DESC
    LIMIT 10
""").df()

**Reading the results:** São Paulo (SP) dominates in absolute revenue —
by far the largest customer base — but revenue *per customer* is higher
in several smaller states, partly driven by freight costs and ticket
size. One-off buyers dominate the marketplace: even top customers rarely
exceed a handful of orders, making retention the clearest growth lever.

### 4.4 Does delivery performance affect satisfaction?

In [ ]:
# Review score by delivery delay bucket
con.execute("""
    WITH delivery AS (
        SELECT
            o.order_id,
            DATE_DIFF('day', o.order_estimated_delivery_date,
                      o.order_delivered_customer_date) AS days_vs_estimate
        FROM orders o
        WHERE o.order_status = 'delivered'
          AND o.order_delivered_customer_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN d.days_vs_estimate <= -7 THEN '1. More than a week early'
            WHEN d.days_vs_estimate <= -1 THEN '2. Early'
            WHEN d.days_vs_estimate = 0  THEN '3. On time'
            WHEN d.days_vs_estimate <= 7 THEN '4. Up to a week late'
            ELSE '5. More than a week late'
        END AS delivery_bucket,
        COUNT(*) AS orders,
        ROUND(AVG(r.review_score), 2) AS avg_review_score
    FROM delivery d
    JOIN reviews r ON d.order_id = r.order_id
    GROUP BY 1
    ORDER BY 1
""").df()

**Reading the results:** The relationship is stark. Orders delivered
early or on time average **~4.2–4.3 stars**; orders more than a week
late crash to **~1.7 stars**. Since most orders actually arrive earlier
than the estimate, the estimate itself is conservative — the real
business risk is the late tail. Reducing late deliveries is the single
most direct lever to protect customer satisfaction.

## 5. Key Findings

### Key Findings

**1. Growth with a seasonal engine.** Revenue grew steadily through
2017–2018, with Black Friday (Nov 2017, +49% MoM) acting as both a sales
peak and a customer-acquisition event that lifted the baseline afterwards.

**2. A long-tail marketplace.** No category dominates: the top category
holds ~10% of revenue and the top 10 only ~60%. Growth requires both
strengthening leaders and improving discovery of niche categories.

**3. Retention is the untapped lever.** São Paulo concentrates the
largest customer base, but nearly all customers — even top spenders —
buy only once or twice. Small retention gains would compound quickly.

**4. Late delivery destroys satisfaction.** Early/on-time orders average
4.0–4.3 stars; orders over a week late collapse to 1.7. Most orders
arrive well ahead of the estimate, so the priority is not speed overall
but eliminating the late tail.

### Recommendations

- Plan inventory, logistics and paid traffic around the November peak.
- Launch post-purchase retention campaigns (cross-sell, replenishment
  reminders) to convert one-time buyers into repeat customers.
- Set delivery-time SLAs with sellers and flag at-risk orders early;
  proactive communication on delayed orders can soften review impact.

### Technical Note

Analysis performed in SQL (DuckDB) over 9 relational tables (~100K
orders): multi-table JOINs, CTEs, window functions (`LAG`, `RANK`,
`SUM() OVER`), date arithmetic and conditional bucketing (`CASE WHEN`).